# A — Cleaning & Integration 

This notebook contains the **actual transparent implementation** of the cleaning and integration stage.

It starts from `data/raw/` and creates:

- cleaned training/test plot data
- cleaned weather and price tables
- seasonal weather features
- price enrichment for analysis/demo only
- `data/processed/master_train.csv`
- `data/processed/master_test.csv`
- `data/processed/data_dictionary_master.csv`
- cleaning log, join audit, join proof, integrity checks

**Important:** market price is deliberately not a model feature. The reason is to prevent data leakage, maintain causal direction, and ensure real-world applicability:

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = ROOT / "figures"

PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
MONTH_NUM = {m: i + 1 for i, m in enumerate(MONTHS)}

REGION_MAP = {
    "amhara": "Amhara", "amhar": "Amhara", "amh": "Amhara",
    "oromia": "Oromia", "oro": "Oromia",
    "snnpr": "SNNPR", "snn": "SNNPR", "snnp": "SNNPR",
    "somali": "Somali", "som": "Somali",
    "tigray": "Tigray", "tig": "Tigray",
}
CROP_MAP = {
    "barley": "barley", "maize": "maize", "sorghum": "sorghum",
    "teff": "teff", "tef": "teff", "wheat": "wheat"
}

def clean_labels(df):
    df = df.copy()
    if "region" in df:
        df["region"] = (
            df["region"].astype(str).str.strip().str.lower().map(REGION_MAP)
        )
    if "crop_type" in df:
        df["crop_type"] = (
            df["crop_type"].astype(str).str.strip().str.lower().map(CROP_MAP)
        )
    if "planting_month" in df:
        df["planting_month"] = df["planting_month"].astype(str).str.strip().str.title()
    if "month" in df:
        df["month"] = df["month"].astype(str).str.strip().str.title()
    return df

In [3]:
# Load the raw files exactly as supplied.
train_raw = pd.read_csv(RAW / "crop_yield_train.csv")
test_raw = pd.read_csv(RAW / "crop_yield_leaderboard_test.csv")
weather_raw = pd.read_csv(RAW / "regional_weather.csv")
prices_raw = pd.read_csv(RAW / "market_prices.csv")
submission_template = pd.read_csv(RAW / "submission_template.csv")

print("Raw shapes")
print("train:", train_raw.shape)
print("test:", test_raw.shape)
print("weather:", weather_raw.shape)
print("prices:", prices_raw.shape)
print("submission template:", submission_template.shape)

display(train_raw.head())

Raw shapes
train: (15090, 15)
test: (3750, 14)
weather: (232, 6)
prices: (100, 4)
submission template: (3750, 2)


,plot_id,region,crop_type,survey_year,planting_month,altitude_m,rainfall_mm_season,farm_size_ha,fertilizer_kg_per_ha,improved_seed_used,pest_disease_flag,soil_quality_index,labor_days_per_ha,distance_to_market_km,yield_tons_per_ha
0,PLOT-000339,OROMIA,barley,2024,Aug,2697.817349,1157.403560,0.867604,29.393684,0,1,0.399408,42.678102,8.573726,2.209711
1,PLOT-007457,Amhara,Maize,2023,Jun,1523.350141,897.420122,1.099197,16.843992,1,0,0.314488,45.370495,19.928302,3.363254
2,PLOT-006092,OROMIA,wheat,2024,Feb,2389.441948,1119.155741,0.811309,41.015144,0,0,0.812596,52.846126,8.961797,3.560614
3,PLOT-004128,SNNPR,barley,2022,Aug,2612.483420,1246.462210,1.602312,14.353759,0,1,0.619516,59.833780,1.849243,1.924800
4,PLOT-003033,Amhara,sorghum,2021,Jul,1210.095040,928.477088,0.937758,33.736138,1,0,0.577963,22.020843,3.149654,4.171220


In [3]:
# Standardize labels before making joins.
train = clean_labels(train_raw)
test = clean_labels(test_raw)
weather = clean_labels(weather_raw)
prices = clean_labels(prices_raw)

print("Canonical regions:", sorted(train["region"].dropna().unique()))
print("Canonical crops:", sorted(train["crop_type"].dropna().unique()))
print("Canonical planting months:", sorted(train["planting_month"].dropna().unique()))

Canonical regions: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray']
Canonical crops: ['barley', 'maize', 'sorghum', 'teff', 'wheat']
Canonical planting months: ['Aug', 'Feb', 'Jul', 'Jun', 'Mar']


In [4]:
# Audit the duplicated plot records.
# The hackathon data contains 90 duplicate-content records.
duplicate_subset = [c for c in train.columns if c != "plot_id"]
duplicate_mask = train.duplicated(subset=duplicate_subset, keep=False)

print("Duplicate-content rows:", duplicate_mask.sum())
print(train.loc[duplicate_mask, ["plot_id", "region", "crop_type", "survey_year"]].head(20))

# Keep one copy of each identical observation.
train = train.drop_duplicates(subset=duplicate_subset, keep="first").copy()
print("Train shape after duplicate-content removal:", train.shape)

Duplicate-content rows: 180
              plot_id  region crop_type  survey_year
2         PLOT-006092  Oromia     wheat         2024
25    PLOT-007993-DUP   SNNPR   sorghum         2024
82    PLOT-006125-DUP   SNNPR      teff         2022
128   PLOT-002713-DUP  Tigray    barley         2023
164   PLOT-009941-DUP  Somali     maize         2024
199       PLOT-008749  Tigray      teff         2022
212   PLOT-005366-DUP   SNNPR     wheat         2022
267   PLOT-003941-DUP  Tigray     wheat         2021
327       PLOT-000099  Tigray     maize         2024
369   PLOT-008982-DUP  Amhara      teff         2022
480   PLOT-008994-DUP  Somali      teff         2024
628   PLOT-013015-DUP  Somali    barley         2024
896       PLOT-007095  Somali   sorghum         2023
903   PLOT-011302-DUP  Oromia   sorghum         2024
932   PLOT-014171-DUP   SNNPR   sorghum         2024
1025      PLOT-011136  Somali     wheat         2023
1160      PLOT-011091  Somali     wheat         2021
1255      PLOT-004

In [5]:
# Replace invalid sentinel values with NaN.
# -999 is not a valid value for these fields.
sentinel_columns = ["pest_disease_flag", "labor_days_per_ha"]

for col in sentinel_columns:
    train[col] = train[col].replace(-999, np.nan)
    test[col] = test[col].replace(-999, np.nan)

print("Missing values after sentinel replacement:")
display(train[sentinel_columns].isna().sum().to_frame("missing"))

Missing values after sentinel replacement:


,missing
pest_disease_flag,297
labor_days_per_ha,483


In [6]:
# Capping obvious high-end unit anomalies using thresholds calculated from TRAIN ONLY.
# These thresholds are calculated before duplicate removal to match the packaged methodology.
farm_cap = train_raw["farm_size_ha"].quantile(0.99)
fertilizer_cap = train_raw["fertilizer_kg_per_ha"].quantile(0.99)

print("Farm-size 99th percentile:", farm_cap)
print("Fertilizer 99th percentile:", fertilizer_cap)

for df in [train, test]:
    df.loc[df["farm_size_ha"] > farm_cap, "farm_size_ha"] = farm_cap
    df.loc[df["fertilizer_kg_per_ha"] > fertilizer_cap, "fertilizer_kg_per_ha"] = fertilizer_cap

Farm-size 99th percentile: 4.262495503276447
Fertilizer 99th percentile: 136.04985626055972


In [7]:
# Clean weather:
# 1) standardize region/month labels
# 2) collapse duplicate region-year-month keys by averaging readings
# 3) impute missing temperature/rainfall by cleaned region-month median
# 4) fall back to global median

weather = (
    weather.groupby(["region", "year", "month"], as_index=False)[
        ["avg_temp_c", "monthly_rainfall_mm", "extreme_heat_days"]
    ].mean()
)

for col in ["avg_temp_c", "monthly_rainfall_mm"]:
    weather[col] = weather[col].fillna(
        weather.groupby(["region", "month"])[col].transform("median")
    )
    weather[col] = weather[col].fillna(weather[col].median())

weather["extreme_heat_days"] = weather["extreme_heat_days"].fillna(
    weather["extreme_heat_days"].median()
)

print("Clean weather shape:", weather.shape)
print("Duplicate seasonal keys:", weather.duplicated(["region", "year", "month"]).sum())
display(weather.head())

Clean weather shape: (226, 6)
Duplicate seasonal keys: 0


,region,year,month,avg_temp_c,monthly_rainfall_mm,extreme_heat_days
0,Amhara,2021,Apr,18.5,102.2,0.0
1,Amhara,2021,Aug,15.7,285.8,0.0
2,Amhara,2021,Dec,16.9,35.7,0.0
3,Amhara,2021,Feb,17.0,77.4,0.0
4,Amhara,2021,Jan,17.3,31.6,1.0


In [8]:
# Clean market prices.
# Values below 100 are treated as the known unit anomaly and multiplied by 100.
prices.loc[
    prices["price_birr_per_quintal"] < 100,
    "price_birr_per_quintal"
] *= 100

# Missing prices are filled using the crop-year median across regions.
prices["price_birr_per_quintal"] = prices["price_birr_per_quintal"].fillna(
    prices.groupby(["crop_type", "year"])["price_birr_per_quintal"].transform("median")
)
prices["price_birr_per_quintal"] = prices["price_birr_per_quintal"].fillna(
    prices["price_birr_per_quintal"].median()
)

print("Remaining missing prices:", prices["price_birr_per_quintal"].isna().sum())
display(prices.head())

Remaining missing prices: 0


,crop_type,region,year,price_birr_per_quintal
0,sorghum,Amhara,2022,3039.0
1,wheat,SNNPR,2023,5321.0
2,maize,Amhara,2022,2824.0
3,wheat,Tigray,2022,4686.0
4,sorghum,Somali,2023,3642.0


In [9]:
# Build four-month seasonal weather features.
# The window is: planting month + next three calendar months.

def seasonal_weather_for_row(row, weather_table):
    start = MONTH_NUM[row["planting_month"]]
    season_months = [MONTHS[(start - 1 + i) % 12] for i in range(4)]

    w = weather_table[
        (weather_table["region"] == row["region"]) &
        (weather_table["year"] == int(row["survey_year"])) &
        (weather_table["month"].isin(season_months))
    ].copy()

    return pd.Series({
        "season_mean_temp": w["avg_temp_c"].mean(),
        "season_rainfall_mm": w["monthly_rainfall_mm"].sum(),
        "season_extreme_heat_days": w["extreme_heat_days"].sum(),
        "weather_months_observed": len(w),
        "season_temp_std": w["avg_temp_c"].std(ddof=0),
        "season_rainfall_std": w["monthly_rainfall_mm"].std(ddof=0),
    })

for df in [train, test]:
    seasonal = df.apply(
        seasonal_weather_for_row,
        axis=1,
        weather_table=weather
    )
    for col in seasonal.columns:
        df[col] = seasonal[col].values

display(train[[
    "region", "crop_type", "survey_year", "planting_month",
    "season_mean_temp", "season_rainfall_mm",
    "season_extreme_heat_days", "weather_months_observed"
]].head())

,region,crop_type,survey_year,planting_month,season_mean_temp,season_rainfall_mm,season_extreme_heat_days,weather_months_observed
0,Oromia,barley,2024,Aug,17.866667,353.0,0.0,3.0
1,Amhara,maize,2023,Jun,15.200000,709.4,0.0,4.0
2,Oromia,wheat,2024,Feb,19.200000,294.9,0.0,4.0
3,SNNPR,barley,2022,Aug,19.425000,398.3,1.0,4.0
4,Amhara,sorghum,2021,Jul,16.500000,798.2,1.0,4.0


In [10]:
# Calculate regional seasonal temperature climatology.
# For every region/year/start-month, compute the mean temperature of the available
# four-month growing-season window, then average those seasonal values by region.

seasonal_climatology_rows = []

for region in sorted(weather["region"].dropna().unique()):
    years = sorted(weather.loc[weather["region"] == region, "year"].unique())
    for year in years:
        for start_month, start_number in MONTH_NUM.items():
            season_months = [MONTHS[(start_number - 1 + i) % 12] for i in range(4)]
            w = weather[
                (weather["region"] == region) &
                (weather["year"] == year) &
                (weather["month"].isin(season_months))
            ]
            if len(w):
                seasonal_climatology_rows.append({
                    "region": region,
                    "year": year,
                    "planting_month": start_month,
                    "season_mean_temp": w["avg_temp_c"].mean()
                })

seasonal_climatology = pd.DataFrame(seasonal_climatology_rows)
region_temperature_climatology = (
    seasonal_climatology.groupby("region")["season_mean_temp"].mean()
)

print("Regional seasonal temperature climatology:")
display(region_temperature_climatology.to_frame("region_avg_temp"))

Regional seasonal temperature climatology:


,region_avg_temp
region,
Amhara,17.105208
Oromia,19.047569
SNNPR,20.450347
Somali,28.663542
Tigray,18.082465


In [11]:
# Add model features.
# Price is intentionally NOT included here.

for df in [train, test]:
    df["season_temp_anomaly"] = (
        df["season_mean_temp"] -
        df["region"].map(region_temperature_climatology)
    )
    df["planting_month_num"] = df["planting_month"].map(MONTH_NUM)
    df["fertilizer_seed_interaction"] = (
        df["fertilizer_kg_per_ha"] * df["improved_seed_used"]
    )
    df["rainfall_reporting_ratio"] = (
        df["rainfall_mm_season"] / df["season_rainfall_mm"]
    )
    df["temp_rainfall_interaction"] = (
        df["season_mean_temp"] *
        np.log1p(df["season_rainfall_mm"])
    )

feature_columns = [
    "survey_year", "altitude_m", "rainfall_mm_season",
    "farm_size_ha", "fertilizer_kg_per_ha", "improved_seed_used",
    "pest_disease_flag", "soil_quality_index", "labor_days_per_ha",
    "distance_to_market_km", "season_mean_temp", "season_rainfall_mm",
    "season_extreme_heat_days", "weather_months_observed",
    "season_temp_std", "season_rainfall_std", "planting_month_num",
    "fertilizer_seed_interaction", "rainfall_reporting_ratio",
    "temp_rainfall_interaction", "season_temp_anomaly"
]

print("Model numeric features:", len(feature_columns))
display(train[feature_columns].head())

Model numeric features: 21


,survey_year,altitude_m,rainfall_mm_season,farm_size_ha,fertilizer_kg_per_ha,improved_seed_used,pest_disease_flag,soil_quality_index,labor_days_per_ha,distance_to_market_km,season_mean_temp,season_rainfall_mm,season_extreme_heat_days,weather_months_observed,season_temp_std,season_rainfall_std,planting_month_num,fertilizer_seed_interaction,rainfall_reporting_ratio,temp_rainfall_interaction,season_temp_anomaly
0,2024,2697.817349,1157.403560,0.867604,29.393684,0,1.0,0.399408,42.678102,8.573726,17.866667,353.0,0.0,3.0,1.960159,70.229924,8,0.000000,3.278764,104.864772,-1.180903
1,2023,1523.350141,897.420122,1.099197,16.843992,1,0.0,0.314488,45.370495,19.928302,15.200000,709.4,0.0,4.0,1.055936,46.886592,6,16.843992,1.265041,99.800589,-1.905208
2,2024,2389.441948,1119.155741,0.811309,41.015144,0,0.0,0.812596,52.846126,8.961797,19.200000,294.9,0.0,4.0,0.951315,13.493031,2,0.000000,3.795035,109.248414,0.152431
3,2022,2612.483420,1246.462210,1.602312,14.353759,0,1.0,0.619516,59.833780,1.849243,19.425000,398.3,1.0,4.0,1.294942,81.580892,8,0.000000,3.129456,116.350175,-1.025347
4,2021,1210.095040,928.477088,0.937758,33.736138,1,0.0,0.577963,22.020843,3.149654,16.500000,798.2,1.0,4.0,1.288410,103.747783,7,33.736138,1.163214,110.279585,-0.605208


In [12]:
# Add price only to a separate analysis/demo copy.
price_for_merge = prices.rename(columns={"year": "survey_year"})

train_analysis = train.merge(
    price_for_merge,
    on=["region", "crop_type", "survey_year"],
    how="left",
    validate="many_to_one"
)

test_analysis = test.merge(
    price_for_merge,
    on=["region", "crop_type", "survey_year"],
    how="left",
    validate="many_to_one"
)

print("Analysis train shape:", train_analysis.shape)
print("Price missing after merge:", train_analysis["price_birr_per_quintal"].isna().sum())

Analysis train shape: (15000, 27)
Price missing after merge: 0


In [13]:
# Save the master tables.
train.to_csv(PROCESSED / "master_train.csv", index=False)
test.to_csv(PROCESSED / "master_test.csv", index=False)

# Keep cleaned weather and prices for reproducible downstream work.
weather.to_csv(PROCESSED / "cleaned_weather.csv", index=False)
prices.to_csv(PROCESSED / "cleaned_prices.csv", index=False)

print("Saved processed tables.")

Saved processed tables.


In [14]:
# Build a data dictionary directly from the resulting master training table.
dictionary_rows = []

for col in train.columns:
    series = train[col]
    dictionary_rows.append({
        "column": col,
        "dtype": str(series.dtype),
        "missing_count": int(series.isna().sum()),
        "unique_count": int(series.nunique(dropna=True)),
        "description": "Target yield" if col == "yield_tons_per_ha"
                      else "Identifier" if col == "plot_id"
                      else "Feature / derived feature"
    })

data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary.to_csv(PROCESSED / "data_dictionary_master.csv", index=False)
display(data_dictionary)

,column,dtype,missing_count,unique_count,description
0,plot_id,object,0,15000,Identifier
1,region,object,0,5,Feature / derived feature
2,crop_type,object,0,5,Feature / derived feature
3,survey_year,int64,0,4,Feature / derived feature
4,planting_month,object,0,5,Feature / derived feature
5,altitude_m,float64,0,14397,Feature / derived feature
6,rainfall_mm_season,float64,447,14546,Feature / derived feature
7,farm_size_ha,float64,0,14850,Feature / derived feature
8,fertilizer_kg_per_ha,float64,742,14116,Feature / derived feature
9,improved_seed_used,int64,0,2,Feature / derived feature


In [15]:
# Produce the cleaning log used by the final report.
cleaning_log = pd.DataFrame([
    {
        "file":"crop_yield_train.csv",
        "column(s)":"region/crop_type",
        "issue_type":"inconsistent labels",
        "rows_affected":0,
        "fix_applied":"strip/lower/map to canonical labels",
        "why":"ensures identical join/category keys"
    },
    {
        "file":"crop_yield_train.csv",
        "column(s)":"plot_id + all fields",
        "issue_type":"duplicate plot record",
        "rows_affected":int(duplicate_mask.sum() // 2),
        "fix_applied":"remove duplicate-content plot records",
        "why":"preserves unique training observations"
    },
    {
        "file":"crop_yield_train.csv",
        "column(s)":"pest_disease_flag, labor_days_per_ha",
        "issue_type":"-999 sentinel",
        "rows_affected":int((train_raw[["pest_disease_flag","labor_days_per_ha"]].eq(-999)).sum().sum()),
        "fix_applied":"replace -999 with missing; impute during model preprocessing",
        "why":"-999 is not a valid domain value"
    },
    {
        "file":"crop_yield_train.csv",
        "column(s)":"farm_size_ha",
        "issue_type":"extreme/unit anomaly",
        "rows_affected":int((train_raw["farm_size_ha"] > farm_cap).sum()),
        "fix_applied":f"cap above train 99th percentile ({farm_cap:.3f} ha)",
        "why":"limits implausible high-end values"
    },
    {
        "file":"crop_yield_train.csv",
        "column(s)":"fertilizer_kg_per_ha",
        "issue_type":"extreme/unit anomaly",
        "rows_affected":int((train_raw["fertilizer_kg_per_ha"] > fertilizer_cap).sum()),
        "fix_applied":f"cap above train 99th percentile ({fertilizer_cap:.3f} kg/ha)",
        "why":"limits obvious high-end unit anomalies"
    },
    {
        "file":"regional_weather.csv",
        "column(s)":"region",
        "issue_type":"inconsistent labels/abbreviations",
        "rows_affected":int(weather_raw["region"].nunique() - weather["region"].nunique()),
        "fix_applied":"map full names and abbreviations to canonical labels",
        "why":"makes region join keys consistent"
    },
    {
        "file":"regional_weather.csv",
        "column(s)":"region/year/month",
        "issue_type":"duplicate weather keys",
        "rows_affected":int(weather_raw.duplicated(["region","year","month"]).sum()),
        "fix_applied":"average duplicate readings by key",
        "why":"restores one row per region-year-month"
    },
    {
        "file":"regional_weather.csv",
        "column(s)":"avg_temp_c/monthly_rainfall_mm",
        "issue_type":"missing weather readings",
        "rows_affected":int(weather_raw[["avg_temp_c","monthly_rainfall_mm"]].isna().sum().sum()),
        "fix_applied":"region-month median then global median",
        "why":"keeps seasonal features computable"
    },
    {
        "file":"market_prices.csv",
        "column(s)":"price_birr_per_quintal",
        "issue_type":"wrong unit",
        "rows_affected":int((prices_raw["price_birr_per_quintal"] < 100).sum()),
        "fix_applied":"multiply values below 100 by 100",
        "why":"restores intended birr/quintal scale"
    }
])
cleaning_log.to_csv(REPORTS / "cleaning_log_transparent.csv", index=False)
display(cleaning_log)

,file,column(s),issue_type,rows_affected,fix_applied,why
0,crop_yield_train.csv,region/crop_type,inconsistent labels,0,strip/lower/map to canonical labels,ensures identical join/category keys
1,crop_yield_train.csv,plot_id + all fields,duplicate plot record,90,remove duplicate-content plot records,preserves unique training observations
2,crop_yield_train.csv,"pest_disease_flag, labor_days_per_ha",-999 sentinel,788,replace -999 with missing; impute during model...,-999 is not a valid domain value
3,crop_yield_train.csv,farm_size_ha,extreme/unit anomaly,151,cap above train 99th percentile (4.262 ha),limits implausible high-end values
4,crop_yield_train.csv,fertilizer_kg_per_ha,extreme/unit anomaly,143,cap above train 99th percentile (136.050 kg/ha),limits obvious high-end unit anomalies
5,regional_weather.csv,region,inconsistent labels/abbreviations,15,map full names and abbreviations to canonical ...,makes region join keys consistent
6,regional_weather.csv,region/year/month,duplicate weather keys,6,average duplicate readings by key,restores one row per region-year-month
7,regional_weather.csv,avg_temp_c/monthly_rainfall_mm,missing weather readings,3,region-month median then global median,keeps seasonal features computable
8,market_prices.csv,price_birr_per_quintal,wrong unit,4,multiply values below 100 by 100,restores intended birr/quintal scale


In [16]:
# Join audit.
join_audit = pd.DataFrame([
    {
        "join":"plot -> weather seasonal",
        "join_type":"many-to-one",
        "rows_before":len(train),
        "rows_after":len(train),
        "match_rate_pct":100.0,
        "unmatched_plots":int(train["season_mean_temp"].isna().sum()),
        "fewer_than_4_months":int((train["weather_months_observed"] < 4).sum()),
        "handling":"aggregate available months; retain observed-month count"
    },
    {
        "join":"plot -> price",
        "join_type":"many-to-one",
        "rows_before":len(train),
        "rows_after":len(train_analysis),
        "match_rate_pct":100.0 * train_analysis["price_birr_per_quintal"].notna().mean(),
        "unmatched_plots":int(train_analysis["price_birr_per_quintal"].isna().sum()),
        "fewer_than_4_months":np.nan,
        "handling":"cleaned/imputed price table; merge validate=many_to_one"
    }
])
join_audit.to_csv(REPORTS / "join_audit_transparent.csv", index=False)
display(join_audit)

,join,join_type,rows_before,rows_after,match_rate_pct,unmatched_plots,fewer_than_4_months,handling
0,plot -> weather seasonal,many-to-one,15000,15000,100.0,0,2905.0,aggregate available months; retain observed-mo...
1,plot -> price,many-to-one,15000,15000,100.0,0,NaN,cleaned/imputed price table; merge validate=ma...


In [17]:
# Integrity checks.
checks = []

def check(name, condition):
    checks.append(("PASS" if bool(condition) else "FAIL", name))

check("plot_id unique in master train", train["plot_id"].is_unique)
check("train row count preserved after duplicate removal", len(train) == 15000)
check("test row count preserved", len(test) == len(test_raw))
check("region labels canonical", set(train["region"].dropna().unique()) <= set(REGION_MAP.values()))
check("crop labels canonical", set(train["crop_type"].dropna().unique()) <= set(CROP_MAP.values()))
check("yield non-negative", (train["yield_tons_per_ha"] >= 0).all())
check("price positive", (prices["price_birr_per_quintal"] > 0).all())
check("no price in model feature list", "price_birr_per_quintal" not in feature_columns)
check("submission IDs unique", submission_template["plot_id"].is_unique)
check("weather seasonal keys unique", not weather.duplicated(["region","year","month"]).any())
check("no missing target", not train["yield_tons_per_ha"].isna().any())

integrity = pd.DataFrame(checks, columns=["status","check"])
print(integrity.to_string(index=False))
integrity.to_csv(REPORTS / "integrity_checks_transparent.csv", index=False)
assert (integrity["status"] == "PASS").all(), "At least one integrity check failed."

status                                             check
  PASS                    plot_id unique in master train
  PASS train row count preserved after duplicate removal
  PASS                          test row count preserved
  PASS                           region labels canonical
  PASS                             crop labels canonical
  PASS                                yield non-negative
  PASS                                    price positive
  PASS                    no price in model feature list
  PASS                             submission IDs unique
  PASS                      weather seasonal keys unique
  PASS                                 no missing target


In [18]:
# Create a small, human-readable join proof using three real plots.
proof = []

for plot_id in train["plot_id"].head(3):
    row = train.loc[train["plot_id"] == plot_id].iloc[0]
    start = MONTH_NUM[row["planting_month"]]
    season_months = [MONTHS[(start - 1 + i) % 12] for i in range(4)]
    w = weather[
        (weather["region"] == row["region"]) &
        (weather["year"] == int(row["survey_year"])) &
        (weather["month"].isin(season_months))
    ].copy()
    w["plot_id"] = plot_id
    proof.append(w)

join_proof = pd.concat(proof, ignore_index=True)
join_proof.to_csv(REPORTS / "join_proof_3_plots_transparent.csv", index=False)
display(join_proof)

,region,year,month,avg_temp_c,monthly_rainfall_mm,extreme_heat_days,plot_id
0,Oromia,2024,Aug,19.1,185.1,0.0,PLOT-000339
1,Oromia,2024,Nov,19.4,20.8,0.0,PLOT-000339
2,Oromia,2024,Sep,15.1,147.1,0.0,PLOT-000339
3,Amhara,2023,Aug,14.5,192.8,0.0,PLOT-007457
4,Amhara,2023,Jul,14.9,244.6,0.0,PLOT-007457
5,Amhara,2023,Jun,17.0,118.5,0.0,PLOT-007457
6,Amhara,2023,Sep,14.4,153.5,0.0,PLOT-007457
7,Oromia,2024,Apr,20.2,55.8,0.0,PLOT-006092
8,Oromia,2024,Feb,18.2,78.5,0.0,PLOT-006092
9,Oromia,2024,Mar,20.1,92.5,0.0,PLOT-006092


## End of A

At this point the complete data pipeline is visible:

**raw files → label normalization → duplicate handling → sentinel handling → train-only caps → weather cleaning → price cleaning → four-month seasonal aggregation → feature engineering → integrity checks → master tables.**

Run this notebook before Notebook 02.